In [1]:
import os
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document
from uuid import uuid4

In [2]:
docs = [
    Document(page_content="Supervised learning trains models on labeled input-output pairs to predict unseen data.", metadata={"topic": "ml"}),
    Document(page_content="Neural networks learn by adjusting weights through backpropagation and gradient descent.", metadata={"topic": "ml"}),
    Document(page_content="Overfitting occurs when a model memorises training data and performs poorly on new data.", metadata={"topic": "ml"}),
    Document(page_content="Training data quality and quantity are the most important factors in model performance.", metadata={"topic": "ml"}),
    Document(page_content="Cross-validation splits data into folds to evaluate model generalisation more reliably.", metadata={"topic": "ml"}),
    Document(page_content="Inflation is the rate at which the general level of prices for goods and services rises over time.", metadata={"topic": "economics"}),
    Document(page_content="GDP measures the total monetary value of all goods and services produced within a country.", metadata={"topic": "economics"}),
    Document(page_content="Interest rates set by central banks influence borrowing costs and consumer spending.", metadata={"topic": "economics"}),
    Document(page_content="Caramelisation occurs when sugar is heated above 160°C, creating complex flavour compounds.", metadata={"topic": "cooking"}),
    Document(page_content="Fermentation uses microorganisms to convert sugars into alcohol or acids, preserving food.", metadata={"topic": "cooking"}),
    Document(page_content="A marathon is a long-distance race of exactly 42.195 kilometres, run on roads.", metadata={"topic": "sports"}),
    Document(page_content="Tennis scoring follows a love, 15, 30, 40, game sequence, with deuce at 40-40.", metadata={"topic": "sports"}),
]

In [3]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

W1031 15:13:22.482000 41120 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [5]:
vector_store = Chroma(
    embedding_function=embeddings,
    collection_name="demo",
    collection_configuration={
        "hnsw" : {
            "space":"cosine"   # l2=euclidean distance , ip=dot product(inner product)
        }
    }
)

In [6]:
ids_added = vector_store.add_documents(documents=docs,
                                       ids=[str(uuid4()) for _ in range(len(docs))])

In [7]:
print(len(ids_added))

12


In [19]:
query = "How does ML model training work?"
similarity_scores = vector_store.similarity_search_with_score(query,k=5)
for doc,score in similarity_scores:
    print(f"Score:{score:.4f} | Topic:{doc.metadata['topic']} | content:{doc.page_content}")

Score:0.5052 | Topic:ml | content:Training data quality and quantity are the most important factors in model performance.
Score:0.5567 | Topic:ml | content:Neural networks learn by adjusting weights through backpropagation and gradient descent.
Score:0.6047 | Topic:ml | content:Cross-validation splits data into folds to evaluate model generalisation more reliably.
Score:0.6137 | Topic:ml | content:Supervised learning trains models on labeled input-output pairs to predict unseen data.
Score:0.6194 | Topic:ml | content:Overfitting occurs when a model memorises training data and performs poorly on new data.


In [20]:
# The above document score is in ascending order due to the internal implementation
# If you want the actual scores just do (1-score)

for _,score in similarity_scores:
    print(f"similarity_score:{1-score:.4f}")

similarity_score:0.4948
similarity_score:0.4433
similarity_score:0.3953
similarity_score:0.3863
similarity_score:0.3806


In [18]:
query = "How does ML model training works?"

threshold = 0.5

retriever = vector_store.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={"score_threshold":threshold},
)

results = retriever.invoke(query)


In [13]:
print(f"Threshold = {threshold}-{len(results)} document(s) returned")
for i, doc in enumerate(results,1):
    print(f"[{i}] topic={doc.metadata['topic']}:{doc.page_content}")

Threshold = 0.5-1 document(s) returned
[1] topic=ml:Training data quality and quantity are the most important factors in model performance.
